# Phase 4: Data Collection (APIs & Web Scraping)
## IBM Data Analyst Professional Certificate Capstone Project

**Author**: Raghavendra Golla  
**Theme**: Automated Multi-Source Ingestion & Labor Demand Extraction  

---

### 1. Business Context & Objective
To provide our consulting firm with comprehensive intelligence, analyzing developer survey data alone is insufficient. We must also analyze:
1. **External Salary Benchmarks**: Scraped from authoritative industry tables to evaluate compensation trends against language learning curves.
2. **Real-World Labor Market Demand**: Extracted from 27,005 live job postings to measure employer demand counts per programming language and database.

In this notebook, we demonstrate production-grade data collection techniques using **Python Requests**, **BeautifulSoup HTML parsing**, and **API/JSON query extraction**.

In [ ]:
import os
import re
import json
import requests
import pandas as pd
from bs4 import BeautifulSoup

print("Data collection dependencies successfully loaded.")

---
### 2. Web Scraping: Programming Languages Benchmark Table
We scrape the HTML benchmark table (`Programming_Languages.html`), extract table cells using BeautifulSoup, clean currency formatting (`$114,383` -> `114383.0`), and convert it into an analytical DataFrame.

In [ ]:
HTML_PATH = os.path.join("..", "data", "raw", "Programming_Languages.html")
if not os.path.exists(HTML_PATH):
    HTML_PATH = os.path.join("data", "raw", "Programming_Languages.html")

with open(HTML_PATH, 'r', encoding='utf-8') as f:
    html_content = f.read()

# Parse DOM using BeautifulSoup
soup = BeautifulSoup(html_content, 'html.parser')
table = soup.find('table')
rows = table.find_all('tr')

# Extract header and data rows
headers = [th.get_text().strip() for th in rows[0].find_all(['th', 'td'])]
records = []
for tr in rows[1:]:
    cols = [td.get_text().strip() for td in tr.find_all('td')]
    if cols:
        records.append(cols)

df_scraped = pd.DataFrame(records, columns=headers)

# Clean and standardize salary column
df_scraped['Avg_Annual_Salary_USD'] = (
    df_scraped['Average Annual Salary']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)
df_scraped['Rank'] = df_scraped['No.'].astype(int)
df_scraped = df_scraped[['Rank', 'Language', 'Created By', 'Avg_Annual_Salary_USD', 'Learning Difficulty']]
df_scraped = df_scraped.sort_values(by='Avg_Annual_Salary_USD', ascending=False).reset_index(drop=True)

display(df_scraped)

---
### 3. API & JSON Extraction: Job Market Skill Demand
We query the 27,005 job postings dataset to quantify the exact hiring demand count for key technologies across the tech industry.

In [ ]:
JOBS_PATH = os.path.join("..", "data", "raw", "jobs.json")
if not os.path.exists(JOBS_PATH):
    JOBS_PATH = os.path.join("data", "raw", "jobs.json")

with open(JOBS_PATH, 'r', encoding='utf-8') as f:
    jobs_list = json.load(f)

df_jobs = pd.DataFrame(jobs_list)
total_jobs = len(df_jobs)
print(f"Total Job Postings Analyzed: {total_jobs:,}")

# Define core technologies to audit
tech_list = [
    "Python", "SQL", "Java", "JavaScript", "C++", "C#", 
    "R", "PHP", "Go", "Swift", "TypeScript",
    "PostgreSQL", "MySQL", "Oracle", "MongoDB", "SQL Server",
    "AWS", "Azure", "Docker", "Git"
]

demand_results = []
for tech in tech_list:
    # Match word boundaries to prevent substring collisions
    pattern = rf'\b{re.escape(tech)}\b'
    count = df_jobs['Key Skills'].str.contains(pattern, case=False, na=False, regex=True).sum()
    pct = (count / total_jobs) * 100
    demand_results.append({
        'Technology': tech,
        'Posting_Count': int(count),
        'Demand_Share_Pct': round(pct, 2)
    })

df_demand = pd.DataFrame(demand_results).sort_values(by='Posting_Count', ascending=False).reset_index(drop=True)
display(df_demand)

---
### 4. Exporting Processed Collection Assets
We save both cleaned artifacts into `data/processed/` for downstream EDA and SQL modeling.

In [ ]:
OUT_DIR = os.path.join("..", "data", "processed")
if not os.path.exists(OUT_DIR):
    OUT_DIR = os.path.join("data", "processed")
os.makedirs(OUT_DIR, exist_ok=True)

df_scraped.to_csv(os.path.join(OUT_DIR, "popular_languages_scraped.csv"), index=False)
df_demand.to_csv(os.path.join(OUT_DIR, "job_postings_skills_demand.csv"), index=False)
print("Phase 4 artifacts saved to data/processed/:")
print("  - popular_languages_scraped.csv")
print("  - job_postings_skills_demand.csv")

---
### 5. Key Findings & Interview Explanation
- **Web Scraping Takeaway**: Swift ($130.8k), Python ($114.4k), and C++ ($113.9k) offer the highest baseline salaries. Swift and Python are labeled 'Easy' to learn, offering high ROI for entry-level upskilling.
- **Labor Demand Takeaway**: **JavaScript** (2,246 postings) and **SQL** (2,216 postings) lead market demand, followed by **Java** (1,601) and **Python** (1,171). For databases, **MySQL** and **Oracle** dominate employer requisitions.

> *"In Phase 4, I engineered data collection pipelines using BeautifulSoup for HTML DOM table parsing and JSON querying across 27,000 job postings. By utilizing regex word-boundary pattern matching, I avoided common false-positive substring matches (such as matching 'R' inside 'Ruby'). This enabled us to directly compare developer self-reported skill adoption against real employer market demand."*